In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import scanpy as sc
import torch

import numpy as np
import random
from scipy.sparse import issparse
import SpaDiff as sd

In [ ]:
parser = sd.get_args() 
args, unknown = parser.parse_known_args() 
args.n_clusters = 30
args.domains = 30
print(args.random_seed)
torch.manual_seed(args.random_seed)
torch.cuda.manual_seed_all(args.random_seed)
np.random.seed(args.random_seed)
random.seed(args.random_seed)
torch.backends.cudnn.deterministic = True
if args.cuda >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{args.cuda}")
    torch.cuda.manual_seed_all(args.random_seed)
    print(f"Using GPU: {torch.cuda.get_device_name(args.cuda)}")
else:
    device = torch.device("cpu")
    print("Using CPU.")

### --- 1. 加载和预处理数据 ---

In [ ]:
st_samples = ["A","P"]  
adata_A = sc.read_visium('E:/gxy_2/2/0_data/case2/Anterior')
adata_A.var_names_make_unique()
adata_P = sc.read_visium('E:/gxy_2/2/0_data/case2/Posterior')
adata_P.var_names_make_unique()

x_pixel1=adata_A.obsm["spatial"][:,1] 
y_pixel1=adata_A.obsm["spatial"][:,0] 
x_pixel2=adata_P.obsm["spatial"][:,1] 
y_pixel2=adata_P.obsm["spatial"][:,0] 

adata_P.obsm["spatial"][:,1]=x_pixel2-np.min(x_pixel2)+np.min(x_pixel1)
adata_P.obsm["spatial"][:,0]=y_pixel2-np.min(y_pixel2)+np.max(y_pixel1)

adata=adata_A.concatenate(adata_P,join='inner',batch_key="batch_name",batch_categories=st_samples)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, subset=True, flavor="seurat_v3", n_top_genes=3000)

In [ ]:
adata, HL = sd.spatial_rec_multi(adata, normalize_total=True,n_neighbors=args.n_neighbors,alpha=args.rec_alpha)
sc.pp.pca(adata, n_comps=50)
features = torch.tensor(adata.obsm['X_pca'].toarray() if issparse(adata.obsm['X_pca']) else np.array(adata.obsm['X_pca'])).float()

In [ ]:
clf=sd.DEC( features, HL, 
              node_width=features.shape[1],
              device = device,
              opt="adam", 
              args=args)

a = clf.fit(features, HL)
y_pred, prob, z=clf.predict()
adata.obsm['z'] = z

In [ ]:
adata = sd.adjust_louvain_resolution(adata, args.domains, use_rep='z')

In [ ]:
Batch_list = []
for sample in st_samples:
    Batch_list.append(adata[adata.obs['batch_name'] == sample])

In [ ]:
plot_color = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf", "#f2a6d4", "#6a9c0a", "#d1581e", "#3b4e97", "#66b200", "#ff5c8f", 
    "#ffcc00", "#ff8d4d", "#62c4da", "#a7a7a7", "#e1c39b", "#9c6c6c", "#c13b5b", "#5c82b3", 
    "#ba8dff", "#b8c239", "#f2b1d9", "#2c6d8f", "#ff6f61", "#4daf4a", "#cfa250", "#8a2be2", 
    "#e67e22", "#9b59b6", "#d0e2a9", "#67a7e5", "#b8dbf0", "#ffadad", "#ffca3a", "#f3b562", 
    "#94b3b8", "#c5f1f0", "#b46f68", "#f47c7c", "#02c39a", "#7b4b95", "#fc9a8d", "#9d78a1"
]
sc.pl.spatial(adata,
              img_key=None,
              color="louvain",
              title="",
              palette=plot_color,
              spot_size=120,
              legend_loc=None, 
              frameon=False,
              show=False
              )